# DLP Assignment 01 on Google Colab

Optional path: the same pipeline as the command line, with the GPU that Q1 and Q2 need.
All three questions are compulsory. Q3 (RNN/LSTM) also runs fine on a laptop CPU.

**Before you run anything:** Runtime -> Change runtime type -> **T4 GPU**.

Upload the whole `DLP_Assignment_01` folder to your Google Drive first (e.g. to
`MyDrive/DLP_Assignment_01`). Keeping it on Drive means a disconnect does not cost you
the 1.8 GB download again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PACK = '/content/drive/MyDrive/DLP_Assignment_01'   # <-- edit if you put it elsewhere
%cd $PACK
!ls

In [ ]:
# torch and torchvision are already on Colab - install only what is missing
!pip -q install ultralytics segmentation-models-pytorch torchmetrics pycocotools
!python env_check.py

## Datasets

About 1.8 GB in total: VOC 2007 for Q1, Oxford-IIIT Pet for Q2, IMDB for Q3. Run once;
skip on later sessions if `data/` is already on Drive.

In [ ]:
!python setup/download_datasets.py --all
!python setup/voc_to_yolo.py

## Question 1 - object detection: YOLOv8 vs Faster R-CNN

Both detectors read the split `voc_to_yolo.py` just produced. Do not re-run the
converter between them, or the comparison stops being fair. Implement the box-predictor
TODO in `starter/q1b_faster_rcnn.py` before the second cell.

In [ ]:
!python starter/q1a_yolo.py --zero-shot
!python starter/q1a_yolo.py

In [ ]:
!python starter/q1b_faster_rcnn.py

## Question 2 - semantic segmentation

Implement `freeze_encoder` before the second cell, or it will raise
`NotImplementedError` by design.

In [ ]:
!python starter/q2_segmentation.py --model unet
!python starter/q2_segmentation.py --model unet --use-dice
!python starter/q2_segmentation.py --model unet --encoder-weights none

In [ ]:
!python starter/q2_segmentation.py --model unet --freeze-encoder
!python starter/q2_segmentation.py --model deeplabv3_resnet50

In [ ]:
from IPython.display import Image, display
display(Image('runs/q2/unet_imagenet/qualitative.png'))
display(Image('runs/q2/unet_imagenet/curves.png'))

## Question 3 - RNNs and LSTMs

Implement both TODO blocks in `starter/q3_rnn_lstm.py` first: packing the padded batch,
and the mean-pooling readout. Each run is only a few minutes, so do not batch-and-pray -
read the printed `mean grad norm` as the vanilla RNN trains.

In [ ]:
# 3.2 - the three cells at matched hidden size
!python starter/q3_rnn_lstm.py --cell rnn
!python starter/q3_rnn_lstm.py --cell lstm
!python starter/q3_rnn_lstm.py --cell gru

In [ ]:
# 3.3 - the sequence-length study
!python starter/q3_rnn_lstm.py --cell rnn  --max-len 50
!python starter/q3_rnn_lstm.py --cell rnn  --max-len 500
!python starter/q3_rnn_lstm.py --cell lstm --max-len 50
!python starter/q3_rnn_lstm.py --cell lstm --max-len 500

In [ ]:
# 3.4 - gradient flow: run both, then plot the two profiles on one log axis
!python starter/q3_rnn_lstm.py --cell rnn  --max-len 500 --grad-flow
!python starter/q3_rnn_lstm.py --cell lstm --max-len 500 --grad-flow

In [ ]:
import json, sys
sys.path.insert(0, '.')
from utils.viz import plot_grad_norms

series = {}
for cell in ('rnn', 'lstm'):
    path = 'runs/q3/{}_len500_h128/grad_norms.json'.format(cell)
    series.update(json.load(open(path)))
plot_grad_norms(series, 'runs/q3/grad_norms_rnn_vs_lstm.png')

from IPython.display import Image, display
display(Image('runs/q3/grad_norms_rnn_vs_lstm.png'))

## Tables for the report

Run this last, after every experiment you intend to report.

In [ ]:
!python starter/make_tables.py
print(open('report/tables.md', encoding='utf-8').read())